In [ ]:
from utils import *
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from mace.calculators import MACECalculator

calc = MACECalculator(model_paths="../training/alloys_pes.model", 
                      device="cuda", default_dtype="float64")
calc_raman  = MACECalculator(model_paths="../training/alloys_dielectric.model",
                             model_type="DipoleMACE",device="cuda", default_dtype="float64")

In [ ]:
std = 3
strains = np.linspace(1.02,0.98,5)
x,y = 0,0
xspace = np.linspace(0,500,5000)
fontsize = 15
plt.rcParams['font.size'] = fontsize
fig, axs = plt.subplots(3, 1, figsize=(5, 10.5),sharex='col')
nmodes = 9
# Choose a colormap that reflects strain (inferno, plasma, or hot)
base_cmap = plt.cm.viridis  # Other options: plt.cm.hot, plt.cm.plasma
cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_inferno", base_cmap(np.linspace(0.3, 0.9, 256))  # Clip to avoid the lightest colors
)

norm = plt.Normalize(vmin=min(strains), vmax=max(strains))  # Normalize temperatures


for strain in strains:
    
    color = cmap(norm(strain))
    
    atoms = make_alloy(x,y,1)
    
    #Strain the cell
    atoms.set_cell(atoms.cell*np.array([strain,strain,1.0]),scale_atoms=True)
    #Calculate eigenvectors and frequencies
    sup_eige,frequencies = get_eigenvectors_and_freq(atoms,calc)

    #Effective Raman
    Reff = get_effective_raman(x,y)
    effective_ramans_mace = get_sc_tensors_effective(sup_eige,Reff)
    effective_raman_intensities = get_raman_intensities(effective_ramans_mace)
    xmace_eff,ymace_eff = get_spectra_curve(xspace,effective_raman_intensities,frequencies,std)
    axs[2].plot(xmace_eff,ymace_eff,label=strain,color=color)
    axs[2].set_yticks([])
    axs[2].legend(frameon=False,fontsize=fontsize-5)

    #Direct Raman Prediction
    ramans_mace = get_sc_tensors(sup_eige,atoms,calc_raman)
    raman_intensities = get_raman_intensities(ramans_mace)
    xmace,ymace = get_spectra_curve(xspace,raman_intensities,frequencies,std)
    axs[1].plot(xmace,ymace,label=strain,color=color)
    axs[1].set_yticks([])
    axs[1].xaxis.set_minor_locator(MultipleLocator(2))
    axs[1].legend(frameon=False,fontsize=fontsize-5)

    # DFT Read
    filename = f'dft_strain_raman/x{x:.3f}y{y:.3f}s{strain:.3f}.dm.out'
    freqs,intens = read_dft_spectra(filename,nmodes)
    xdft,ydft = get_spectra_curve(xspace,intens,freqs,std)
    axs[0].plot(xdft,ydft,label=strain,color=color)
    axs[0].set_yticks([])
    axs[0].xaxis.set_minor_locator(MultipleLocator(2))
    axs[0].legend(frameon=False,fontsize=fontsize-5)
    
axs[0].text(0.01,0.95,'(a)',transform=axs[0].transAxes,ha='left',va='top')
axs[1].text(0.01,0.95,'(b)',transform=axs[1].transAxes,ha='left',va='top')
axs[2].text(0.01,0.95,'(c)',transform=axs[2].transAxes,ha='left',va='top')
plt.xlim(340,420)
plt.subplots_adjust(hspace=0.05)

fig.text(0.5, 0.072, r"Raman shift (cm$^{-1}$)", ha='center',fontsize=fontsize)
fig.text(0.05, 0.5, "Intensity (a.u.)", va='center', rotation='vertical',fontsize=fontsize)

plt.savefig('raman_spectra_vs_strain.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)
plt.show()